# M05 — Optimization from Scratch

**Optimization is the engine under every ML algorithm.** When you understand *how* models are trained — not just that they are trained — you can diagnose failures, tune hyperparameters intelligently, and explain your choices to a technical panel.

## What you will build

Every optimizer in this notebook is implemented from scratch using only numpy. You will verify each against scipy or a known reference.

## The core concept: gradient descent family

All gradient-based optimizers follow the same skeleton:
$$\theta^{(t+1)} = \theta^{(t)} - \eta \cdot \tilde{g}^{(t)}$$

where $\tilde{g}^{(t)}$ is some transformation of the gradient $g^{(t)} = \nabla_\theta \mathcal{L}$. The optimizer defines how $\tilde{g}$ is computed:

| Optimizer | $\tilde{g}$ | Key idea |
|---|---|---|
| SGD | $g$ | Raw gradient |
| Momentum | $v = \beta v + g$ | Accumulate velocity |
| RMSProp | $g / \sqrt{\mathbb{E}[g^2]}$ | Normalize by recent gradient magnitude |
| Adam | Momentum + RMSProp (with bias correction) | Best of both |

**Reference:** [numpy docs](https://numpy.org/doc/stable/)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification, make_regression, fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# Synthetic datasets
np.random.seed(42)
X_cls, y_cls = make_classification(n_samples=3000, n_features=10, n_informative=6, random_state=42)
X_reg, y_reg = make_regression(n_samples=3000, n_features=10, noise=0.5, random_state=42)

X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42)
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

sc = StandardScaler()
X_tr_c = sc.fit_transform(X_tr_c); X_te_c = sc.transform(X_te_c)
sc2 = StandardScaler()
X_tr_r = sc2.fit_transform(X_tr_r); X_te_r = sc2.transform(X_te_r)

# Add bias column
def add_bias(X): return np.hstack([X, np.ones((X.shape[0], 1))])
Xb_tr_c = add_bias(X_tr_c); Xb_te_c = add_bias(X_te_c)
Xb_tr_r = add_bias(X_tr_r); Xb_te_r = add_bias(X_te_r)

def sigmoid(z): return np.where(z >= 0, 1/(1+np.exp(-z)), np.exp(z)/(1+np.exp(z)))
def bce_loss(y, p): p = np.clip(p, 1e-9, 1-1e-9); return -np.mean(y*np.log(p) + (1-y)*np.log(1-p))
def bce_grad(X, y, beta): p = sigmoid(X @ beta); return X.T @ (p - y) / len(y)
def mse_loss(y, yhat): return np.mean((y - yhat)**2)
def mse_grad(X, y, beta): return X.T @ (X @ beta - y) * 2 / len(y)

print(f"Train: {X_tr_c.shape} | Features + bias: {Xb_tr_c.shape[1]}")

---
## Exercise 1 — SGD with Learning Rate Schedules

## The Math

Plain SGD with a fixed learning rate often oscillates near the minimum. **Learning rate decay** fixes this:

- **Step decay:** $\eta_t = \eta_0 \cdot \gamma^{\lfloor t/s \rfloor}$ — reduce by factor $\gamma$ every $s$ steps
- **Exponential decay:** $\eta_t = \eta_0 \cdot e^{-kt}$
- **Cosine annealing:** $\eta_t = \eta_{\min} + \frac{1}{2}(\eta_{\max} - \eta_{\min})(1 + \cos(\pi t / T))$
- **Warmup + decay:** Linear increase from 0 to $\eta_0$ for first $w$ steps, then decay

**Task:** Implement `SGDOptimizer` supporting all four schedules. Apply to logistic regression on the classification dataset. Compare convergence speed (loss vs epoch) for each schedule.

In [ ]:
class SGDOptimizer:
    """
    Stochastic Gradient Descent with learning rate schedules.
    """
    def __init__(self, lr: float = 0.1, schedule: str = 'constant',
                 decay: float = 0.95, step_size: int = 10,
                 warmup_steps: int = 5, n_epochs: int = 100,
                 batch_size: int = 64, random_state: int = 42):
        self.lr0 = lr
        self.schedule = schedule   # 'constant', 'step', 'exponential', 'cosine', 'warmup'
        self.decay = decay
        self.step_size = step_size
        self.warmup_steps = warmup_steps
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.random_state = random_state

    def get_lr(self, epoch: int) -> float:
        """
        Return current learning rate based on schedule and epoch.
        """
        # YOUR CODE HERE
        pass

    def optimize(self, X: np.ndarray, y: np.ndarray,
                  grad_fn, loss_fn) -> tuple:
        """
        Run SGD with mini-batches.
        Returns (final_beta, loss_history, lr_history)
        """
        # YOUR CODE HERE
        pass

# Test all 4 schedules
results = {}
for sched in ['constant', 'step', 'exponential', 'cosine']:
    opt = SGDOptimizer(lr=0.5, schedule=sched, n_epochs=80, batch_size=64)
    beta, losses, lrs = opt.optimize(Xb_tr_c, y_tr_c, bce_grad, bce_loss)
    if beta is not None:
        test_auc = roc_auc_score(y_te_c, sigmoid(Xb_te_c @ beta))
        results[sched] = {'final_loss': losses[-1], 'test_auc': test_auc, 'lr_history': lrs}
        print(f"{sched:12s}: loss={losses[-1]:.4f}, auc={test_auc:.4f}")

In [ ]:
# --- ASSERTIONS ---
assert len(results) == 4
for sched, res in results.items():
    assert res['test_auc'] > 0.65, f"{sched}: AUC too low"
    assert res['final_loss'] < 0.8, f"{sched}: loss too high"

# LR schedule sanity checks
opt_cos = SGDOptimizer(lr=0.5, schedule='cosine', n_epochs=100)
lrs = [opt_cos.get_lr(t) for t in range(100)]
assert lrs[0] > lrs[-1], "Cosine: lr must decrease overall"
assert lrs[50] < lrs[0], "Cosine: lr must drop by midpoint"

opt_step = SGDOptimizer(lr=0.5, schedule='step', decay=0.5, step_size=10)
lr0 = opt_step.get_lr(0); lr10 = opt_step.get_lr(10)
assert abs(lr10 - lr0 * 0.5) < 1e-9, "Step decay: lr must halve at step 10"

print("✓ Exercise 1 passed")

---
## Exercise 2 — Momentum

## The Math

Momentum accumulates a velocity vector $v$ in the direction of persistent gradients:
$$v^{(t)} = \beta v^{(t-1)} + g^{(t)}$$
$$\theta^{(t+1)} = \theta^{(t)} - \eta v^{(t)}$$

**Intuition:** Think of a ball rolling down a hill. It accelerates in consistent gradient directions and dampens oscillations in noisy directions. $\beta$ (typically 0.9) controls how much of the previous velocity survives.

**Nesterov Momentum** (a stronger variant) computes the gradient at the *lookahead* position:
$$v^{(t)} = \beta v^{(t-1)} + g(\theta^{(t)} - \eta \beta v^{(t-1)})$$

The lookahead corrects the momentum direction before applying it — faster convergence.

**Task:** Implement both standard and Nesterov momentum. Compare convergence to plain SGD.

In [ ]:
class MomentumOptimizer:
    """
    SGD with Momentum (standard and Nesterov).
    """
    def __init__(self, lr: float = 0.1, beta: float = 0.9,
                 nesterov: bool = False, n_epochs: int = 100,
                 batch_size: int = 64, random_state: int = 42):
        self.lr = lr
        self.beta = beta
        self.nesterov = nesterov
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.random_state = random_state

    def optimize(self, X: np.ndarray, y: np.ndarray,
                  grad_fn, loss_fn) -> tuple:
        """
        Returns (final_beta, loss_history)
        Standard: v = beta*v + grad; theta -= lr * v
        Nesterov: grad at theta - lr*beta*v; then standard update
        """
        # YOUR CODE HERE
        pass

sgd_beta, sgd_losses, _ = SGDOptimizer(lr=0.1, n_epochs=100, batch_size=64).optimize(
    Xb_tr_c, y_tr_c, bce_grad, bce_loss)
mom_beta, mom_losses = MomentumOptimizer(lr=0.05, beta=0.9, nesterov=False, n_epochs=100).optimize(
    Xb_tr_c, y_tr_c, bce_grad, bce_loss)
nest_beta, nest_losses = MomentumOptimizer(lr=0.05, beta=0.9, nesterov=True, n_epochs=100).optimize(
    Xb_tr_c, y_tr_c, bce_grad, bce_loss)

In [ ]:
# --- ASSERTIONS ---
for beta, name, losses in [
    (mom_beta, 'Momentum', mom_losses),
    (nest_beta, 'Nesterov', nest_losses)
]:
    assert beta is not None, f"{name}: beta is None"
    assert losses[-1] < losses[0], f"{name}: loss must decrease"
    auc = roc_auc_score(y_te_c, sigmoid(Xb_te_c @ beta))
    assert auc > 0.65, f"{name}: AUC {auc:.4f} too low"
    print(f"{name:10s}: final_loss={losses[-1]:.4f}, auc={auc:.4f}")

# Momentum should converge faster than plain SGD
# (lower loss at epoch 50 with same LR, adjusting for scale)
print(f"SGD loss @50:  {sgd_losses[49]:.4f}")
print(f"Momentum @50:  {mom_losses[49]:.4f}")
print(f"Nesterov @50:  {nest_losses[49]:.4f}")
print("✓ Exercise 2 passed")

---
## Exercise 3 — RMSProp

## The Math

RMSProp adapts the learning rate per parameter by dividing by a running average of squared gradients:

$$E[g^2]^{(t)} = \rho E[g^2]^{(t-1)} + (1-\rho) g^{(t)^2}$$
$$\theta^{(t+1)} = \theta^{(t)} - \frac{\eta}{\sqrt{E[g^2]^{(t)} + \epsilon}} g^{(t)}$$

**Intuition:** Features with large, frequent gradients get a small effective learning rate. Features with small, rare gradients get a large effective learning rate. This is especially useful for sparse features (e.g., embeddings).

**$\epsilon$** (typically $10^{-8}$) prevents division by zero.

**Task:** Implement `RMSPropOptimizer`. Show that it handles features with very different gradient scales better than plain SGD.

In [ ]:
class RMSPropOptimizer:
    """
    RMSProp: adaptive per-parameter learning rates.
    E[g^2] = rho * E[g^2] + (1-rho) * g^2
    theta -= lr / sqrt(E[g^2] + eps) * g
    """
    def __init__(self, lr: float = 0.01, rho: float = 0.9,
                 eps: float = 1e-8, n_epochs: int = 100,
                 batch_size: int = 64, random_state: int = 42):
        self.lr = lr
        self.rho = rho
        self.eps = eps
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.random_state = random_state

    def optimize(self, X: np.ndarray, y: np.ndarray,
                  grad_fn, loss_fn) -> tuple:
        """
        Returns (final_beta, loss_history, effective_lr_history)
        effective_lr_history: mean effective LR per epoch = lr / sqrt(E[g^2] + eps)
        """
        # YOUR CODE HERE
        pass

rmsprop_beta, rmsprop_losses, eff_lrs = RMSPropOptimizer(
    lr=0.01, rho=0.9, n_epochs=100
).optimize(Xb_tr_c, y_tr_c, bce_grad, bce_loss)

In [ ]:
# --- ASSERTIONS ---
assert rmsprop_beta is not None
assert rmsprop_losses[-1] < rmsprop_losses[0], "RMSProp loss must decrease"
auc = roc_auc_score(y_te_c, sigmoid(Xb_te_c @ rmsprop_beta))
assert auc > 0.65
# Effective LR should vary per parameter (adaptive)
if eff_lrs is not None and len(eff_lrs) > 0:
    assert len(eff_lrs) == 100
print(f"✓ Exercise 3 passed — RMSProp AUC: {auc:.4f}, final_loss: {rmsprop_losses[-1]:.4f}")

---
## Exercise 4 — Adam: The Standard in Practice

## The Math

Adam combines momentum (first moment) with RMSProp (second moment), plus **bias correction**:

**First moment (momentum):**
$$m^{(t)} = \beta_1 m^{(t-1)} + (1-\beta_1) g^{(t)}$$

**Second moment (RMSProp):**
$$v^{(t)} = \beta_2 v^{(t-1)} + (1-\beta_2) g^{(t)^2}$$

**Bias correction** (crucial for early steps when $m$ and $v$ are initialized at 0):
$$\hat{m}^{(t)} = \frac{m^{(t)}}{1 - \beta_1^t}, \quad \hat{v}^{(t)} = \frac{v^{(t)}}{1 - \beta_2^t}$$

**Update:**
$$\theta^{(t+1)} = \theta^{(t)} - \frac{\eta}{\sqrt{\hat{v}^{(t)}} + \epsilon} \hat{m}^{(t)}$$

**Default hyperparameters** ($\beta_1=0.9$, $\beta_2=0.999$, $\epsilon=10^{-8}$, $\eta=10^{-3}$) work well for most problems.

**Why bias correction?** At $t=1$ with $\beta_1=0.9$: $m^{(1)} = 0.1 g^{(1)}$ — severely underestimates the true gradient. Dividing by $(1-0.9^1) = 0.1$ corrects this back to $g^{(1)}$.

**Task:** Implement `AdamOptimizer`. Verify bias correction matters by comparing with and without it.

In [ ]:
class AdamOptimizer:
    """
    Adam optimizer.
    Default: beta1=0.9, beta2=0.999, eps=1e-8, lr=1e-3
    """
    def __init__(self, lr: float = 1e-3, beta1: float = 0.9,
                 beta2: float = 0.999, eps: float = 1e-8,
                 bias_correction: bool = True,
                 n_epochs: int = 100, batch_size: int = 64,
                 random_state: int = 42):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.bias_correction = bias_correction
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.random_state = random_state

    def optimize(self, X: np.ndarray, y: np.ndarray,
                  grad_fn, loss_fn) -> tuple:
        """
        Returns (final_beta, loss_history)
        """
        # YOUR CODE HERE
        # Initialize m=0, v=0, t=0
        # Per update step:
        #   t += 1
        #   m = beta1*m + (1-beta1)*g
        #   v = beta2*v + (1-beta2)*g^2
        #   if bias_correction: m_hat = m/(1-beta1^t), v_hat = v/(1-beta2^t)
        #   else: m_hat=m, v_hat=v
        #   theta -= lr * m_hat / (sqrt(v_hat) + eps)
        pass

adam_beta, adam_losses = AdamOptimizer(lr=0.01, n_epochs=100).optimize(
    Xb_tr_c, y_tr_c, bce_grad, bce_loss)
adam_no_bc_beta, adam_no_bc_losses = AdamOptimizer(lr=0.01, n_epochs=100, bias_correction=False).optimize(
    Xb_tr_c, y_tr_c, bce_grad, bce_loss)

In [ ]:
# --- ASSERTIONS ---
assert adam_beta is not None
assert adam_losses[-1] < adam_losses[0]
auc_adam = roc_auc_score(y_te_c, sigmoid(Xb_te_c @ adam_beta))
assert auc_adam > 0.70, f"Adam AUC too low: {auc_adam:.4f}"

# Bias correction should help especially at early steps
if adam_no_bc_losses is not None:
    early_adam = np.mean(adam_losses[:10])
    early_no_bc = np.mean(adam_no_bc_losses[:10])
    print(f"Early loss (with BC): {early_adam:.4f} | (without BC): {early_no_bc:.4f}")

print(f"✓ Exercise 4 passed — Adam AUC: {auc_adam:.4f}, final_loss: {adam_losses[-1]:.4f}")

---
## Exercise 5 — Optimizer Convergence Comparison

**Task:** Run all 4 optimizers (SGD, Momentum, RMSProp, Adam) on the same regression problem and compare:
1. Loss vs epoch curves
2. Final RMSE on test set
3. Time to converge (first epoch where loss < threshold)
4. Sensitivity to learning rate: for each optimizer, try 3 LR values and record final loss

Return `convergence_df`: optimizer × metric comparison DataFrame.

In [ ]:
# YOUR CODE HERE
convergence_df = None

In [ ]:
# --- ASSERTIONS ---
assert convergence_df is not None
assert len(convergence_df) == 4
for col in ['final_rmse', 'epochs_to_converge']:
    assert col in convergence_df.columns, f"Missing: {col}"
assert (convergence_df['final_rmse'] > 0).all()
print("✓ Exercise 5 passed")
print(convergence_df.to_string())

---
## Exercise 6 — Regularization as Optimization Constraint

## The Math

**L2 regularization (Ridge / Weight Decay):**
$$\mathcal{L}_{\text{reg}} = \mathcal{L} + \frac{\lambda}{2}\|\beta\|^2$$
Gradient: $\nabla_{\text{reg}} = \nabla \mathcal{L} + \lambda \beta$ — pushes weights toward 0.

**L1 regularization (Lasso):**
$$\mathcal{L}_{\text{reg}} = \mathcal{L} + \lambda\|\beta\|_1$$
Gradient: $\nabla_{\text{reg}} = \nabla \mathcal{L} + \lambda \cdot \text{sign}(\beta)$ — pushes weights exactly to 0 (sparsity).

**Why L1 produces sparsity but L2 doesn't:**
The L1 ball is diamond-shaped with corners on the axes. The optimal solution tends to land on a corner where one or more weights are exactly 0. The L2 ball is smooth — no such corners.

**Task:** Implement regularized gradient updates in Adam. Show the sparsity effect of L1 vs the shrinkage effect of L2.

In [ ]:
class RegularizedAdamOptimizer(AdamOptimizer):
    """
    Adam with L1 or L2 regularization.
    """
    def __init__(self, *args, reg_type: str = 'l2', lam: float = 0.01, **kwargs):
        super().__init__(*args, **kwargs)
        self.reg_type = reg_type  # 'l1', 'l2', or 'none'
        self.lam = lam

    def _regularized_grad(self, grad: np.ndarray, beta: np.ndarray) -> np.ndarray:
        """
        Add regularization term to gradient.
        L2: grad + lam * beta
        L1: grad + lam * sign(beta)
        Do NOT regularize the bias term (last element).
        """
        # YOUR CODE HERE
        pass

    def optimize(self, X, y, grad_fn, loss_fn):
        """
        Override optimize to use regularized gradients.
        """
        # YOUR CODE HERE
        pass

# Train all 3 variants
no_reg_beta, _ = AdamOptimizer(lr=0.01, n_epochs=150).optimize(Xb_tr_r, y_tr_r, mse_grad, mse_loss)
l2_beta, _ = RegularizedAdamOptimizer(lr=0.01, n_epochs=150, reg_type='l2', lam=0.1).optimize(
    Xb_tr_r, y_tr_r, mse_grad, mse_loss)
l1_beta, _ = RegularizedAdamOptimizer(lr=0.01, n_epochs=150, reg_type='l1', lam=0.1).optimize(
    Xb_tr_r, y_tr_r, mse_grad, mse_loss)

In [ ]:
# --- ASSERTIONS ---
for beta, name in [(no_reg_beta,'no_reg'), (l2_beta,'l2'), (l1_beta,'l1')]:
    if beta is not None:
        rmse = np.sqrt(mse_loss(y_te_r, Xb_te_r @ beta))
        print(f"{name:8s}: RMSE={rmse:.4f}, max|coef|={np.abs(beta[:-1]).max():.4f}")

# L1 must produce more zeros (sparsity)
if l1_beta is not None and no_reg_beta is not None:
    n_zeros_l1 = (np.abs(l1_beta[:-1]) < 0.01).sum()
    n_zeros_none = (np.abs(no_reg_beta[:-1]) < 0.01).sum()
    print(f"Near-zero coefs: L1={n_zeros_l1}, no_reg={n_zeros_none}")

# L2 must shrink coefficients vs no regularization
if l2_beta is not None and no_reg_beta is not None:
    assert np.abs(l2_beta[:-1]).mean() < np.abs(no_reg_beta[:-1]).mean(), \
        "L2 must shrink coefficients vs no regularization"

print("✓ Exercise 6 passed")

---
## Exercise 7 — Newton's Method and Quasi-Newton

## The Math

Gradient descent uses only first-order information (gradient). Newton's method uses second-order (Hessian):
$$\theta^{(t+1)} = \theta^{(t)} - H^{-1} g^{(t)}$$

where $H = \nabla^2 \mathcal{L}$ is the Hessian matrix.

**Advantage:** Quadratic convergence near the optimum — far fewer iterations than GD.
**Problem:** $H^{-1}$ costs $O(p^3)$ to compute — infeasible for high-dimensional models.

**For logistic regression**, the Hessian has a closed form:
$$H = \frac{1}{n} X^T W X \quad \text{where} \quad W = \text{diag}(p_i(1-p_i))$$

This is the **IRLS (Iteratively Reweighted Least Squares)** algorithm — used internally by statsmodels.

**Task:** Implement Newton's method for logistic regression. Show that it converges in far fewer iterations than gradient descent.

In [ ]:
class NewtonLogisticRegression:
    """
    Logistic regression via Newton-Raphson (IRLS).
    Converges in O(10) iterations vs O(100s) for GD.
    """
    def __init__(self, max_iter: int = 20, tol: float = 1e-6,
                 reg_lambda: float = 1e-4):
        self.max_iter = max_iter
        self.tol = tol
        self.reg_lambda = reg_lambda  # L2 ridge to stabilize Hessian
        self.beta_ = None
        self.loss_history_ = []
        self.n_iter_ = 0

    def fit(self, X: np.ndarray, y: np.ndarray) -> 'NewtonLogisticRegression':
        """
        IRLS update:
        1. p = sigmoid(X @ beta)
        2. W = diag(p * (1-p))  -- n x n diagonal matrix
        3. gradient = X.T @ (p - y) / n
        4. hessian = X.T @ W @ X / n  +  lambda * I  (regularized)
        5. beta -= H^{-1} @ gradient
        6. Stop when ||gradient|| < tol
        """
        # YOUR CODE HERE
        # Hint: don't form the full n×n W matrix — use broadcasting:
        # X.T @ W @ X = (X * p*(1-p)[:, None]).T @ X
        pass

    def predict_proba(self, X):
        p = sigmoid(X @ self.beta_)
        return np.column_stack([1-p, p])

    def predict(self, X):
        return (sigmoid(X @ self.beta_) >= 0.5).astype(int)

newton = NewtonLogisticRegression(max_iter=30, reg_lambda=1e-3)
newton.fit(Xb_tr_c, y_tr_c)

In [ ]:
# --- ASSERTIONS ---
assert newton.beta_ is not None
assert newton.n_iter_ <= 30

auc_newton = roc_auc_score(y_te_c, newton.predict_proba(Xb_te_c)[:, 1])
assert auc_newton > 0.70, f"Newton AUC: {auc_newton:.4f}"

# Newton should converge MUCH faster than GD
# Compare: GD needs 100 epochs, Newton needs < 20 iterations
print(f"Newton iterations: {newton.n_iter_}")
print(f"Newton AUC: {auc_newton:.4f}")
if len(newton.loss_history_) > 1:
    print(f"Loss: {newton.loss_history_[0]:.4f} → {newton.loss_history_[-1]:.4f}")

print("✓ Exercise 7 passed")

---
## Exercise 8 — Hyperparameter Optimization: Grid Search from Scratch

## The Math

Hyperparameter optimization is itself an optimization problem — but we can't differentiate through the cross-validation loop. Common strategies:

- **Grid search:** exhaustive, guaranteed to find the best in the grid, $O(N^k)$ evaluations
- **Random search:** sample from distributions, often finds good solutions in fewer evaluations
- **Bayesian optimization:** build a surrogate model of the CV loss surface, sample where improvement is expected

**Task:** Implement `GridSearchCV` from scratch (no sklearn). Must support:
- Any estimator with `fit()` and `predict_proba()` methods
- K-fold cross-validation
- Any scoring function
- Return results as a DataFrame sorted by mean_test_score

In [ ]:
from itertools import product

class GridSearchCVScratch:
    """
    Manual K-fold cross-validated grid search.
    Works with any estimator implementing fit() and predict_proba().
    """
    def __init__(self, estimator_class, param_grid: dict,
                 cv: int = 5, scoring=None, random_state: int = 42):
        self.estimator_class = estimator_class
        self.param_grid = param_grid  # dict: {param_name: [values]}
        self.cv = cv
        self.scoring = scoring or roc_auc_score
        self.random_state = random_state
        self.results_ = None
        self.best_params_ = None
        self.best_score_ = None

    def _k_fold_split(self, n: int, k: int):
        """
        Yield (train_idx, val_idx) tuples for k-fold CV.
        """
        # YOUR CODE HERE
        pass

    def fit(self, X: np.ndarray, y: np.ndarray) -> 'GridSearchCVScratch':
        """
        Exhaustive grid search with k-fold CV.
        Stores results_ DataFrame.
        """
        # YOUR CODE HERE
        # For each combination of hyperparameters:
        #   For each fold:
        #     train estimator, evaluate on val fold
        #   Record mean and std of CV scores
        pass

# Wrap our LogisticRegressionScratch to accept **kwargs in constructor
# (We'll use sklearn's LogisticRegression as a proxy for testing)
from sklearn.linear_model import LogisticRegression as SklearnLR

param_grid = {'C': [0.01, 0.1, 1.0, 10.0], 'max_iter': [200]}
gs = GridSearchCVScratch(SklearnLR, param_grid, cv=5)
gs.fit(X_tr_c, y_tr_c)

In [ ]:
# --- ASSERTIONS ---
assert gs.results_ is not None
assert len(gs.results_) == 4  # 4 C values
assert 'mean_test_score' in gs.results_.columns
assert 'std_test_score' in gs.results_.columns
assert gs.results_['mean_test_score'].is_monotonic_decreasing
assert gs.best_params_ is not None
assert gs.best_score_ > 0.65
print(f"✓ Exercise 8 passed — Best C={gs.best_params_.get('C')}, AUC={gs.best_score_:.4f}")
print(gs.results_.to_string(index=False))

---
## Exercise 9 — Numerical Gradient Checking

## The Math

When you implement a custom gradient, how do you know it's correct? **Numerical gradient checking** approximates the gradient using finite differences:

$$\frac{\partial \mathcal{L}}{\partial \theta_j} \approx \frac{\mathcal{L}(\theta + \epsilon e_j) - \mathcal{L}(\theta - \epsilon e_j)}{2\epsilon}$$

where $e_j$ is the unit vector for dimension $j$. This is the **centered finite difference** approximation — accurate to $O(\epsilon^2)$.

**Relative error check:**
$$\text{error} = \frac{\|g_{\text{analytic}} - g_{\text{numeric}}\|}{\|g_{\text{analytic}}\| + \|g_{\text{numeric}}\|}$$

If error $< 10^{-5}$: your gradient is correct. If error $> 10^{-3}$: something is wrong.

**Task:** Implement `gradient_check(loss_fn, grad_fn, X, y, beta, eps=1e-5)`. Verify the BCE and MSE gradients we've been using.

In [ ]:
def gradient_check(loss_fn, grad_fn, X: np.ndarray, y: np.ndarray,
                    beta: np.ndarray, eps: float = 1e-5) -> dict:
    """
    Numerically verify analytical gradient.
    Returns dict: analytic_grad, numeric_grad, relative_error, passed (bool)
    """
    # YOUR CODE HERE
    # 1. Compute analytic gradient: grad_fn(X, y, beta)
    # 2. For each dimension j:
    #    beta_plus = beta.copy(); beta_plus[j] += eps
    #    beta_minus = beta.copy(); beta_minus[j] -= eps
    #    numeric_grad[j] = (loss_fn(y, X@beta_plus) - loss_fn(y, X@beta_minus)) / (2*eps)
    # 3. Compute relative error
    pass

# Check BCE gradient
beta_test = np.zeros(Xb_tr_c.shape[1])
bce_check = gradient_check(
    lambda y, yhat: bce_loss(y, sigmoid(yhat)),  # loss expects raw scores
    bce_grad,
    Xb_tr_c[:100], y_tr_c[:100], beta_test
)

# Check MSE gradient
mse_check = gradient_check(
    lambda y, yhat: mse_loss(y, yhat),
    mse_grad,
    Xb_tr_r[:100], y_tr_r[:100], beta_test
)

In [ ]:
# --- ASSERTIONS ---
for check, name in [(bce_check, 'BCE'), (mse_check, 'MSE')]:
    if check is not None:
        assert 'relative_error' in check
        assert 'passed' in check
        print(f"{name} gradient check: error={check['relative_error']:.2e}, passed={check['passed']}")
        if check['relative_error'] is not None:
            assert check['relative_error'] < 1e-4, \
                f"{name} gradient error too high: {check['relative_error']:.2e}"

print("✓ Exercise 9 passed")

---
## Exercise 10 — Capstone: Neural Network from Scratch

## The Math

A 2-layer neural network:
$$z^{[1]} = X W^{[1]} + b^{[1]}, \quad a^{[1]} = \text{ReLU}(z^{[1]})$$
$$z^{[2]} = a^{[1]} W^{[2]} + b^{[2]}, \quad \hat{y} = \sigma(z^{[2]})$$

**Backpropagation** (chain rule):
$$\frac{\partial \mathcal{L}}{\partial z^{[2]}} = \hat{y} - y \quad (\text{BCE gradient})$$
$$\frac{\partial \mathcal{L}}{\partial W^{[2]}} = a^{[1]T} \cdot \frac{\partial \mathcal{L}}{\partial z^{[2]}} / n$$
$$\frac{\partial \mathcal{L}}{\partial a^{[1]}} = \frac{\partial \mathcal{L}}{\partial z^{[2]}} \cdot W^{[2]T}$$
$$\frac{\partial \mathcal{L}}{\partial z^{[1]}} = \frac{\partial \mathcal{L}}{\partial a^{[1]}} \cdot \mathbf{1}[z^{[1]} > 0] \quad (\text{ReLU derivative})$$
$$\frac{\partial \mathcal{L}}{\partial W^{[1]}} = X^T \cdot \frac{\partial \mathcal{L}}{\partial z^{[1]}} / n$$

**Task:** Implement a 2-layer neural network with ReLU + sigmoid using your Adam optimizer. Beat logistic regression on the classification dataset.

In [ ]:
class TwoLayerNeuralNet:
    """
    2-layer NN: Linear -> ReLU -> Linear -> Sigmoid
    Trained with Adam optimizer via backprop.
    """
    def __init__(self, hidden_size: int = 32, lr: float = 1e-3,
                 n_epochs: int = 200, batch_size: int = 64,
                 random_state: int = 42):
        self.hidden_size = hidden_size
        self.lr = lr
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.random_state = random_state
        self.params_ = {}
        self.loss_history_ = []

    def _init_params(self, n_features: int):
        """
        He initialization for ReLU networks:
        W ~ N(0, sqrt(2/n_in))
        b = 0
        """
        # YOUR CODE HERE
        pass

    def _forward(self, X: np.ndarray) -> tuple:
        """
        Forward pass.
        Returns (y_hat, cache) where cache stores z1, a1, z2 for backprop.
        """
        # YOUR CODE HERE
        pass

    def _backward(self, X: np.ndarray, y: np.ndarray, cache: dict) -> dict:
        """
        Backpropagation.
        Returns gradients dict: dW1, db1, dW2, db2
        """
        # YOUR CODE HERE
        pass

    def fit(self, X: np.ndarray, y: np.ndarray) -> 'TwoLayerNeuralNet':
        """
        Train with Adam optimizer and mini-batch gradient descent.
        """
        # YOUR CODE HERE
        # Initialize Adam moments for each parameter
        # Mini-batch loop: forward -> loss -> backward -> Adam update
        pass

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        y_hat, _ = self._forward(X)
        return np.column_stack([1 - y_hat, y_hat])

    def predict(self, X: np.ndarray) -> np.ndarray:
        return (self._forward(X)[0] >= 0.5).astype(int)

nn = TwoLayerNeuralNet(hidden_size=32, lr=1e-3, n_epochs=150)
nn.fit(X_tr_c, y_tr_c)

In [ ]:
# --- ASSERTIONS ---
assert len(nn.params_) >= 4  # W1, b1, W2, b2
assert len(nn.loss_history_) > 0
assert nn.loss_history_[-1] < nn.loss_history_[0], "Loss must decrease"

proba_nn = nn.predict_proba(X_te_c)[:, 1]
auc_nn = roc_auc_score(y_te_c, proba_nn)
assert auc_nn > 0.70, f"NN AUC too low: {auc_nn:.4f}"

# Compare to logistic regression
from sklearn.linear_model import LogisticRegression
lr_ref = LogisticRegression(max_iter=1000, random_state=42).fit(X_tr_c, y_tr_c)
auc_lr = roc_auc_score(y_te_c, lr_ref.predict_proba(X_te_c)[:, 1])

# Run gradient check on the network
W1_shape = nn.params_['W1'].shape
print(f"✓ Exercise 10 passed — NN AUC: {auc_nn:.4f} | LR AUC: {auc_lr:.4f}")
print(f"Architecture: input={X_tr_c.shape[1]} -> hidden={nn.hidden_size} -> output=1")
print(f"Loss: {nn.loss_history_[0]:.4f} → {nn.loss_history_[-1]:.4f}")